<a href="https://colab.research.google.com/github/Kretaceous/NLP-Week1-Text-Classification/blob/main/movie_review_sentiment_classifier_with_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

Hello, I'm **Wesley**, nice to meet you!👋

I was just reading the IMDb reviews of [*The Super Mario Bros. Movie*](https://www.imdb.com/title/tt6718170/), I thought why don't we make a **Sentiment Classifier** to categorize movie reviews! **WARNING: Spoilers ahead.**


Here we will be doing [transfer learning](https://en.wikipedia.org/wiki/Transfer_learning) on BERT [(blog)](https://ai.googleblog.com/2018/11/open-sourcing-bert-state-of-art-pre.html) [(paper)](https://arxiv.org/abs/1810.04805v2) with an IMDb dataset to make a sentiment classifier for movie reviews.

# Setup Python Libraries (pip)

In [1]:
#install some Python packages with pip

!pip install numpy torch datasets transformers evaluate --quiet


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\eshan\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
# let's check the version we are using

!pip freeze | grep -E '^numpy|^torch|^datasets|^transformers|^evaluate'

'grep' is not recognized as an internal or external command,
operable program or batch file.


# Create IMDB Dataset for Fine-tuning BERT

## Load the extracted dataset if it already exists


In [5]:
import pandas as pd
from datasets import Dataset, DatasetDict

extracted_df = pd.read_csv('extracted_toxicity_dataset.csv')
print(f"Loaded 'extracted_toxicity_dataset.csv' into extracted_df. Shape: {extracted_df.shape}")

# Convert the processed pandas DataFrame to a Hugging Face Dataset object.
if not extracted_df.empty:
    hf_dataset = Dataset.from_pandas(extracted_df)

    # Reassign 'raw_dataset' to a DatasetDict containing the processed data.
    global raw_dataset # Declare global to modify the variable in the global scope
    raw_dataset = DatasetDict({
        'train': hf_dataset
    })

    print("\n'raw_dataset' has been updated to a DatasetDict containing the processed data:")
    print(raw_dataset)
else:
    print("'raw_dataset' could not be created as extracted_df is empty.")

Loaded 'extracted_toxicity_dataset.csv' into extracted_df. Shape: (48630, 4)

'raw_dataset' has been updated to a DatasetDict containing the processed data:
DatasetDict({
    train: Dataset({
        features: ['text', 'target_group', 'toxic', 'source'],
        num_rows: 48630
    })
})


## ONLY RUN IF EXTRACTED DATASET NOT ALREADY CREATED:
## Create the train, validation, test sets

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

# The current value of 'raw_dataset' is the CSV file path.
# We'll use this path to load and process the CSV.
file_path_to_process = 'combined_toxicity_dataset.csv'
print(f"Attempting to process CSV file from: {file_path_to_process}")

try:
    # Read the CSV file into a DataFrame
    df = pd.read_csv(file_path_to_process)
    print(f"Successfully loaded CSV. Original DataFrame shape: {df.shape}")

    # Filter for 'jigsaw_human' source
    jigsaw_df = df[df['source'] == 'jigsaw_human']
    print(f"Rows with 'jigsaw_human' source: {len(jigsaw_df)}")

    # Filter for 'toxigen_human_annotated' source
    toxigen_human_annotated_df = df[df['source'] == 'toxigen_human_annotated']
    print(f"Rows with 'toxigen_human_annotated' source: {len(toxigen_human_annotated_df)}")

    sampled_dfs = []

    # Include all 'toxigen_human_annotated' rows
    if not toxigen_human_annotated_df.empty:
        sampled_dfs.append(toxigen_human_annotated_df)
        print(f"Included all {len(toxigen_human_annotated_df)} rows from 'toxigen_human_annotated'.")

        # The number of rows to sample from jigsaw_human will be double the toxigen count
        rows_to_sample_from_jigsaw = 4 * len(toxigen_human_annotated_df)
    else:
        print("Warning: No 'toxigen_human_annotated' rows found. Cannot create a 1:4 split.")
        rows_to_sample_from_jigsaw = 0

    # Sample from jigsaw_human to maintain the 1:4 ratio
    if rows_to_sample_from_jigsaw > 0 and len(jigsaw_df) >= rows_to_sample_from_jigsaw:
        sampled_jigsaw = jigsaw_df.sample(n=rows_to_sample_from_jigsaw, random_state=42)
        sampled_dfs.append(sampled_jigsaw)
        print(f"Sampled {len(sampled_jigsaw)} rows from 'jigsaw_human' (4x 'toxigen_human_annotated').")
    elif rows_to_sample_from_jigsaw > 0:
        print(f"Warning: Not enough 'jigsaw_human' rows ({len(jigsaw_df)}) to sample {rows_to_sample_from_jigsaw}. Taking all available from 'jigsaw_human'.")
        sampled_dfs.append(jigsaw_df)
    else:
        print("No rows to sample from 'jigsaw_human' as 'toxigen_human_annotated' was empty.")

    # Concatenate the sampled DataFrames and shuffle the final result
    if sampled_dfs:
        combined_sampled_df = pd.concat(sampled_dfs).sample(frac=1, random_state=42).reset_index(drop=True)
        extracted_df = combined_sampled_df
        print(f"Combined and shuffled DataFrame shape: {extracted_df.shape}")
        print("\nHead of the extracted DataFrame:")
        print(extracted_df.head())
    else:
        extracted_df = pd.DataFrame() # Empty if no data was sampled
        print("No data could be sampled from the specified sources.")

    # Convert the processed pandas DataFrame to a Hugging Face Dataset object.
    if not extracted_df.empty:
        hf_dataset = Dataset.from_pandas(extracted_df)

        # Reassign 'raw_dataset' to a DatasetDict containing the processed data.
        global raw_dataset # Declare global to modify the variable in the global scope
        raw_dataset = DatasetDict({
            'train': hf_dataset
        })

        print("\n'raw_dataset' has been updated to a DatasetDict containing the processed data:")
        print(raw_dataset)
    else:
        print("'raw_dataset' could not be created as extracted_df is empty.")

except FileNotFoundError:
    print(f"Error: The file '{file_path_to_process}' was not found. Please ensure the path is correct.")
except Exception as e:
    print(f"An error occurred during CSV processing: {e}")

In [ ]:
# Convert extracted data to csv (only need to run if it doesnt already exist!)

output_csv_filename = 'extracted_toxicity_dataset.csv'
extracted_df.to_csv(output_csv_filename, index=False)
print(f"Extracted to '{output_csv_filename}'")

## Split the extracted dataset

In [6]:
from datasets import Dataset, DatasetDict

# Convert extracted_df to a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(extracted_df)

# First, split the main dataset into a training set and a temporary 'eval_set'
# This will result in approximately 80% for training and 20% for the combined validation and test sets.
initial_splits = hf_dataset.train_test_split(test_size=0.2, seed=42, shuffle=True)

# Now, split the 'eval_set' (which is initial_splits['test']) into validation and test sets.
# A 50/50 split of the 20% 'eval_set' will yield approximately 10% for validation and 10% for test of the original data.
validation_test_splits = initial_splits['test'].train_test_split(test_size=0.5, seed=42, shuffle=True)

# Assemble the final dataset dictionary with explicit 'train', 'validation', and 'test' keys
dataset = DatasetDict({
    'train': initial_splits['train'],
    'validation': validation_test_splits['train'],  # The first part of the 20% split becomes validation
    'test': validation_test_splits['test']       # The second part of the 20% split becomes test
})

display(dataset)

DatasetDict({
    train: Dataset({
        features: ['text', 'target_group', 'toxic', 'source'],
        num_rows: 38904
    })
    validation: Dataset({
        features: ['text', 'target_group', 'toxic', 'source'],
        num_rows: 4863
    })
    test: Dataset({
        features: ['text', 'target_group', 'toxic', 'source'],
        num_rows: 4863
    })
})

## We start by tokenizing our dataset with the BERT's Fast Tokenizer

In [7]:
# let's import the pretrained faster tokenizer from huggingface
# source: (https://huggingface.co/distilbert-base-uncased)

from transformers import AutoTokenizer

checkpoint = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
tokenizer

BertTokenizer(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
})

In [8]:
# tokenize the text in batches with truncation and padding based on BERT requirements

def tokenization(example):
    # padding=True was causing errors for mixed lengths, I deleted it so that the Trainer's DataCollator could auto pad it instead
    encoded = tokenizer(example['text'], truncation=True)

    # DistilBERT typically does not use token_type_ids, so we remove them to avoid potential issues with data collation.
    if 'token_type_ids' in encoded:
        del encoded['token_type_ids']
    return encoded

tokenized_dataset = dataset.map(tokenization, batched=True, remove_columns=['target_group', 'source'])

# Rename the 'toxic' column to 'labels' for the Trainer
tokenized_dataset = tokenized_dataset.rename_column('toxic', 'labels')

tokenized_dataset

Map:   0%|          | 0/38904 [00:00<?, ? examples/s]

Map:   0%|          | 0/4863 [00:00<?, ? examples/s]

Map:   0%|          | 0/4863 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 38904
    })
    validation: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 4863
    })
    test: Dataset({
        features: ['text', 'labels', 'input_ids', 'attention_mask'],
        num_rows: 4863
    })
})

# Setup Training Metrics (Accuracy, F1)

In [9]:
import evaluate
import numpy as np

# we setup the training to evaluate the accuracy and f1 scores

accuracy_metric = evaluate.load('accuracy')
f1_metric = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels)
    return {**accuracy, **f1}

# Setup Training Configurations

In [10]:
import os
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

# Install accelerate library to resolve ImportError
#!pip install accelerate

# get bert model with a sequence classification head for sentiment analysis
# source: (https://huggingface.co/distilbert-base-uncased)
checkpoint = 'distilbert-base-uncased'
num_labels = 2
id2label = {0:'NEGATIVE',1:'POSITIVE'}
label2id = {'NEGATIVE':0,'POSITIVE':1}
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=num_labels, id2label=id2label, label2id=label2id)

# setup custom training arguments
# 1. store training checkpoints to 'results' output directory
# 2. fine-tune for just 1 epoch
# 3,4. use 16 as a batch size to speed things up
# 5. evaluate validation set every 500 steps (this is the default steps)
# 6. load the best model based on the lowest validation loss at the end of training
training_args = TrainingArguments(
    seed=42,
    output_dir = './results',
    num_train_epochs = 3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    load_best_model_at_end=True,
    eval_strategy = "epoch",
    save_strategy = 'epoch',
    label_names=['label'] # Explicitly specify the label column for the Trainer
)

# Initialize DataCollatorWithPadding explicitly
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# setup trainer with custom metrics (accuracy, f1)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
    data_collator=data_collator # Pass the data collator explicitly
)

# disable wandb logging (a v4 huggingface artifact)
os.environ['WANDB_DISABLED']= "true"

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


# Evaluate UnFine-Tuned BERT on Test Set for a Baseline Metric


In [26]:
# let's first evaluate unfine-tuned model with test set

trainer.evaluate(tokenized_dataset['test'])

Training Loss,Validation Loss,Epoch,Accuracy,F1
No log,0.695815,0,0.458770,0.412500


{'eval_loss': 0.6958149671554565,
 'eval_accuracy': 0.45877030639522925,
 'eval_f1': 0.4125}

Without fine-tuning BERT, our model currently has around **52% Accuracy (eval_accuracy)** and **19% F1 (eval_f1)**, which is pretty bad due to the test dataset having around 50% positive and 50% negative reviews. 😕


Let's make it better with transfer learning! 🦾

# Fine-Tune BERT with new Toxicity Dataset

In [13]:
import torch
torch.cuda.is_available()

False

In [ ]:
# let's fine-tune BERT with the IMDb dataset

trainer.train()

In [ ]:
# let's see how well it did in the test set

trainer.evaluate(tokenized_dataset['test'])

{'eval_loss': 0.21517841517925262,
 'eval_model_preparation_time': 0.0028,
 'eval_accuracy': 0.92296,
 'eval_f1': 0.9246596776717259,
 'eval_runtime': 393.8639,
 'eval_samples_per_second': 63.474,
 'eval_steps_per_second': 3.968,
 'epoch': 1.0}

**WOAH!** We got a **92% Accuracy (eval_accuracy)** and **92% F1 (eval_f1)** with just **1 epoch**! 🤯

# Try out some examples!

In [ ]:
from transformers import pipeline
import torch

# get current device with pytorch
device = torch.cuda.current_device()

# create pipeline for sentiment classifier with custom model and tokenizer
sentiment_classifier = pipeline(task='sentiment-analysis', model=model, tokenizer=tokenizer, device=device)

In [ ]:
# let's see how our model classifies a good review
# this is from 'justinvitelli' (https://www.imdb.com/review/rw8972952)

review = """
First off this movie is for kids and fans of Nintendo and the Mario franchise.
I still think an adult who isnt a fan could still enjoy it but this movie is so
full of fan service that it will have you smiling the whole time.
The voice acting I was skeptical but they all work and work well too.
Jack Black is the star here. I love how they kept the story simple like all of the games.
Truly felt like a video game on screen.
This movie felt like a beautifully animated amusement park ride.
The audio in the movie was amazing too.
The sounds and the score with reimagined iconic music was perfect.
Some of the songs in the movie felt unnecessary but they worked.
I think they should've bumped the run time to 105-120 min.
90 min felt too short as it goes by quick.
I havent had this much wholesome fun at the movies in a long time.
If youre a fan you HAVE to see it.
"""
sentiment_classifier(review)

[{'label': 'POSITIVE', 'score': 0.9938808679580688}]

That is **99% POSITIVE**! *justinvitelli* loves the movie!

In [ ]:
# let's see how our model classifies a bad review
# this is from 'industriousbug16' (https://www.imdb.com/review/rw8998214)

review = """
Flat, visual noise.
Fundamentally incurious. Potentially injurious.
The mystique generated by the characters in the games is here raked over and presented
haphazardly by hacks.
A hobbled attempt to explain a long and random evolution of characters who were never meant
to be narratised fails.
Doing it well is near impossible when you insist on EVERY LITTLE BIT OF LORE,
from the last forty years being shoehorned into 90 minutes.
Makes little sense, shamelessly leans on member berries to stimulate older viewers but offers
nothing else.
I feel sad for the animators who did a sterling job, but to no end as this movie has no soul.
"""
sentiment_classifier(review)

[{'label': 'NEGATIVE', 'score': 0.9951890707015991}]

That is **99% NEGATIVE**! *industriousbug16* must hate the movie very badly.

# Resources

### If you would like to use this model without running the entire notebook, try the model at my [HuggingFace](https://huggingface.co/wesleyacheng/movie-review-sentiment-classifier-with-bert).

### If you woud like to get this in GitHub, here's my [repo](https://github.com/wesleyacheng/movie-review-sentiment-classifier-with-bert).